# Katabatic — Google Colab Setup

Run this notebook once at the start of each Colab session to mount Drive, clone/update the repo, and install dependencies.

**Hardware:** For GPU-heavy models (CTGAN, GReaT, TabDDPM), use Runtime → Change runtime type → T4 GPU before running.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Clone or update repo from Drive
import os, subprocess, sys

REPO_PATH = '/content/drive/MyDrive/Katabatic'
REPO_URL = 'https://github.com/lukebrumby/katabatic-personal.git'

if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', 'origin', 'luke'], check=True)

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — Install dependencies
# Use requirements.txt for core deps, requirements-all.txt for all model extras
import subprocess, sys

# Change to requirements-all.txt if you need GPU models (ctgan, great, tabddpm, etc.)
req_file = 'requirements.txt'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', req_file],
    check=True
)
print(f'Dependencies installed from {req_file}')

In [ ]:
# Cell 4 — Symlink data directories so existing notebook paths work
import os

for d in ['raw_data', 'discretized_data', 'sample_data', 'Results', 'synthetic']:
    src = f'/content/drive/MyDrive/Katabatic/{d}'
    os.makedirs(src, exist_ok=True)
    if not os.path.exists(d):
        os.symlink(src, d)
        print(f'Symlinked: {d} -> {src}')
    else:
        print(f'Already exists: {d}')

In [ ]:
# Cell 5 — Verify GPU and framework availability
import importlib

# PyTorch GPU check
if importlib.util.find_spec('torch'):
    import torch
    print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('PyTorch: not installed (install requirements-all.txt for GPU models)')

# TensorFlow GPU check
if importlib.util.find_spec('tensorflow'):
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f'TensorFlow: {tf.__version__}, GPUs: {gpus}')
else:
    print('TensorFlow: not installed (install requirements-all.txt for TF models)')

# Katabatic import check
try:
    import katabatic
    print('katabatic: importable')
except ImportError as e:
    print(f'katabatic import failed: {e}')

## Python 3.11 / 3.12 note

Colab defaults to Python 3.12. If TensorFlow fails to install, you can force Python 3.11 via condacolab:

```python
!pip install -q condacolab
import condacolab; condacolab.install_miniforge()
# Runtime restarts automatically — then run:
!conda install -y python=3.11
```

Run that in a separate cell before Cell 3 if you see TensorFlow install errors.